# Grammar Scoring Engine - v3 (improved pipeline)

**What changed versus the previous notebook**

| Problem in v2 | Fix in v3 |
|---|---|
| "Grammar" features were only POS ratios, nothing measured errors | Added **LanguageTool** error counts, **CoLA** acceptability scores, **GPT-2** sentence perplexity |
| Whisper `base` silently corrected speech and lost confidence info | Whisper `small` (configurable) with **word timestamps + per-word probabilities**, segment log-probs, pause/rate stats |
| ~75 mostly-noise acoustic features (chroma, tempo, ...) | Trimmed to a small acoustic set, plus **Whisper-encoder embeddings** (whole clip, 30 s chunks) |
| Trees on 100+ weak features | Added **embedding models** (Ridge/SVR) on Whisper-encoder and sentence-transformer embeddings, TF-IDF Ridge, tabular XGB/ExtraTrees/Ridge |
| Hand-picked 70/20/10 weights | **Stacking** with non-negative linear regression on out-of-fold predictions |
| Final models used different configs than CV; two competing submissions; `final_model` undefined | One consistent flow: each fold model predicts test and predictions are averaged. One submission file |
| Failed audio files were silently dropped (would break submission length) | Failed files keep their row (NaN features, imputed) so the submission is always complete |
| Single 5-fold split | **Repeated stratified 5-fold x 3** |

Runs on Google Colab (GPU strongly recommended). First run is slow (feature extraction is cached to Drive).

## 1. Install dependencies

In [1]:
!pip -q install openai-whisper librosa nltk xgboost soundfile language_tool_python transformers sentence-transformers sentencepiece
!apt-get -qq install -y default-jre-headless > /dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 2.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 3.3 MB/s eta 0:00:00


## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Imports and configuration

In [3]:
import os, re, gc, math, pickle, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import nltk
import torch
import whisper
from tqdm.auto import tqdm

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import RidgeCV, Ridge, LinearRegression
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (RepeatedStratifiedKFold, RepeatedKFold,
                                     GridSearchCV, KFold, cross_val_predict)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# ------------------------- configuration -------------------------
SEED = 42
WHISPER_SIZE = "small"     # "base" | "small" | "medium" | "large-v3"  (bigger = better but slower)
N_SPLITS = 5
N_REPEATS = 3
RUN_DEBERTA = False        # set True to also fine-tune DeBERTa-v3 on transcripts (needs GPU, ~10-20 min)
# ------------------------------------------------------------------

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [4]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
from nltk import word_tokenize, sent_tokenize, pos_tag
print("NLTK ready")

NLTK ready


## 4. Paths and data

In [5]:
DATA_ROOT = Path("/content/drive/MyDrive/data")

TRAIN_CSV = DATA_ROOT / "train.csv"
TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
TRAIN_AUDIO_DIR = DATA_ROOT / "train"
TEST_AUDIO_DIR = DATA_ROOT / "test"

# New cache folder (features changed). Whisper size is in the name so caches never mix.
CACHE_DIR = DATA_ROOT / f"feature_cache_v3_{WHISPER_SIZE}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_CSV)

print("Train:", train_df.shape, "| Test:", test_df.shape, "| Sample sub:", sample_sub.shape)
display(train_df.head())
print(train_df["label"].describe())


def check_audio_files(df, audio_dir, name):
    missing = [f for f in df["filename"].astype(str) if not (Path(audio_dir) / f).exists()]
    print(f"{name}: {len(df) - len(missing)}/{len(df)} files found")
    if missing:
        print("Missing (first 20):", missing[:20])
    return missing

_ = check_audio_files(train_df, TRAIN_AUDIO_DIR, "TRAIN")
_ = check_audio_files(test_df, TEST_AUDIO_DIR, "TEST")

Train: (769, 2) | Test: (216, 2) | Sample sub: (204, 2)


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5
3,audio_126.wav,2.0
4,audio_61.wav,2.0


count    769.000000
mean       3.313394
std        1.239000
min        0.000000
25%        2.500000
50%        3.000000
75%        4.000000
max        5.000000
Name: label, dtype: float64
TRAIN: 769/769 files found
TEST: 216/216 files found


## 5. Load models (Whisper, LanguageTool, CoLA, GPT-2)

In [6]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          GPT2LMHeadModel, GPT2TokenizerFast)

print(f"Loading Whisper {WHISPER_SIZE} ...")
whisper_model = whisper.load_model(WHISPER_SIZE, device=DEVICE)

# LanguageTool needs Java (installed in step 1)
LT_OK = False
try:
    import language_tool_python
    lt_tool = language_tool_python.LanguageTool("en-US")
    lt_tool.check("This is a test sentence.")
    LT_OK = True
    print("LanguageTool ready")
except Exception as e:
    print("LanguageTool NOT available, its features will be skipped:", e)

print("Loading CoLA acceptability model ...")
cola_tok = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
cola_model = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA").to(DEVICE).eval()

print("Loading GPT-2 ...")
gpt_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()
print("All models loaded.")

Loading Whisper small ...


100%|████████████████████████████████████████| 461M/461M [00:02<00:00, 204MiB/s]


LanguageTool ready
Loading CoLA acceptability model ...


config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

All models loaded.


## 6. Feature extraction functions

Feature groups:
1. **Whisper confidence + timing** - word probabilities, segment log-probs, pauses, speech/articulation rate
2. **Text / POS features** - lexical diversity, tense, subordination, sentence structure
3. **Grammar-error features** - LanguageTool, CoLA acceptability, GPT-2 perplexity
4. **Small acoustic set** - energy, pitch, a few MFCC means, voiced fraction
5. **Whisper encoder embedding** (mean + std pooled over the full clip)

In [7]:
# ---------------- 1. Whisper confidence + timing ----------------
FILLERS = {"uh", "um", "erm", "hmm", "er", "ah", "uhm"}


def whisper_features(segments, duration):
    f = {"n_segments": len(segments)}

    if segments:
        lp = np.array([s.get("avg_logprob", np.nan) for s in segments], dtype=float)
        cr = np.array([s.get("compression_ratio", np.nan) for s in segments], dtype=float)
        ns = np.array([s.get("no_speech_prob", np.nan) for s in segments], dtype=float)
        f["seg_logprob_mean"] = np.nanmean(lp)
        f["seg_logprob_min"] = np.nanmin(lp)
        f["seg_compress_max"] = np.nanmax(cr)
        f["seg_nospeech_max"] = np.nanmax(ns)

    words = []
    for s in segments:
        for w in (s.get("words") or []):
            if w.get("start") is not None and w.get("end") is not None:
                words.append(w)
    if not words:
        return f

    probs = np.array([float(w.get("probability", np.nan)) for w in words])
    starts = np.array([float(w["start"]) for w in words])
    ends = np.array([float(w["end"]) for w in words])
    norm = [re.sub(r"[^a-z']", "", w["word"].lower()) for w in words]
    n = len(words)

    f["wp_mean"] = np.nanmean(probs)
    f["wp_std"] = np.nanstd(probs)
    f["wp_min"] = np.nanmin(probs)
    f["wp_p10"] = np.nanpercentile(probs, 10)
    f["wp_p25"] = np.nanpercentile(probs, 25)
    f["wp_frac_lt_05"] = np.nanmean(probs < 0.5)
    f["wp_frac_lt_03"] = np.nanmean(probs < 0.3)

    word_dur = np.clip(ends - starts, 0, None)
    gaps = np.clip(starts[1:] - ends[:-1], 0, None) if n > 1 else np.array([])
    pauses = gaps[gaps >= 0.25]

    f["wd_mean"] = word_dur.mean()
    f["wd_std"] = word_dur.std()
    f["lead_silence"] = starts[0]
    f["trail_silence"] = max(0.0, duration - ends[-1])
    f["speech_span"] = ends[-1] - starts[0]
    f["speech_rate"] = n / max(duration, 1e-6)
    f["articulation_rate"] = n / max(word_dur.sum(), 1e-6)
    f["phonation_ratio"] = word_dur.sum() / max(duration, 1e-6)
    f["n_pauses_025"] = len(pauses)
    f["n_pauses_050"] = int((gaps >= 0.5).sum())
    f["n_pauses_100"] = int((gaps >= 1.0).sum())
    f["pauses_per_word"] = len(pauses) / n
    f["pause_mean"] = pauses.mean() if len(pauses) else 0.0
    f["pause_max"] = pauses.max() if len(pauses) else 0.0
    f["pause_total_ratio"] = pauses.sum() / max(duration, 1e-6)

    f["adj_repeat_words"] = sum(1 for a, b in zip(norm[:-1], norm[1:]) if a and a == b)
    f["adj_repeat_ratio"] = f["adj_repeat_words"] / n
    f["filler_tokens"] = sum(1 for w in norm if w in FILLERS)
    return f

In [8]:
# ---------------- 2. Text / POS features ----------------
POS_TAGS = ["NN", "NNS", "NNP", "VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "MD", "JJ", "JJR", "JJS",
            "RB", "PRP", "PRP$", "DT", "IN", "CC", "TO", "WDT", "WP", "WRB", "EX", "RP", "CD"]
SUBORD = {"because", "although", "though", "while", "whereas", "if", "when", "since", "unless",
          "which", "who", "whom", "whose", "that", "after", "before", "until"}
FILLER_PHRASES = ["like", "actually", "basically", "you know", "i mean", "kind of", "sort of"]


def text_features(text, duration):
    f = {}
    if not isinstance(text, str) or not text.strip():
        f["n_words"] = 0
        return f

    toks = word_tokenize(text)
    tagged = pos_tag(toks)
    alpha = [(w, t) for w, t in tagged if re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", w)]
    n = len(alpha)
    f["n_words"] = n
    if n == 0:
        return f

    words = [w.lower() for w, _ in alpha]
    tags = [t for _, t in alpha]
    tc = Counter(tags)

    for t in POS_TAGS:
        f[f"pos_{t}"] = tc.get(t, 0) / n

    nouns = sum(v for k, v in tc.items() if k.startswith("NN"))
    verbs = sum(v for k, v in tc.items() if k.startswith("VB"))
    adjs = sum(v for k, v in tc.items() if k.startswith("JJ"))
    advs = sum(v for k, v in tc.items() if k.startswith("RB"))
    f["noun_ratio"] = nouns / n
    f["verb_ratio"] = verbs / n
    f["adj_ratio"] = adjs / n
    f["adv_ratio"] = advs / n
    f["verb_noun_ratio"] = verbs / max(nouns, 1)
    f["adj_noun_ratio"] = adjs / max(nouns, 1)
    f["past_verb_share"] = tc.get("VBD", 0) / max(verbs, 1)
    f["pos_diversity"] = len(set(tags)) / n

    trans = list(zip(tags[:-1], tags[1:]))
    f["pos_bigram_diversity"] = len(set(trans)) / max(len(trans), 1)

    uniq = len(set(words))
    f["unique_ratio"] = uniq / n
    f["root_ttr"] = uniq / math.sqrt(n)
    wl = np.array([len(w.replace("'", "")) for w in words])
    f["avg_word_len"] = wl.mean()
    f["long_word_ratio"] = float((wl >= 7).mean())
    f["subord_rate"] = sum(w in SUBORD for w in words) / n

    wc = Counter(words)
    f["repeated_word_ratio"] = sum(c - 1 for c in wc.values() if c > 1) / n

    low = text.lower()
    f["filler_phrase_count"] = sum(len(re.findall(r"\b" + re.escape(p) + r"\b", low)) for p in FILLER_PHRASES)
    f["words_per_sec_text"] = n / max(duration, 1e-6)

    sents = sent_tokenize(text)
    lens = [len(re.findall(r"[A-Za-z']+", s)) for s in sents]
    lens = [l for l in lens if l > 0]
    f["n_sentences"] = len(lens)
    if lens:
        f["sent_len_mean"] = float(np.mean(lens))
        f["sent_len_std"] = float(np.std(lens))
        f["sent_len_max"] = float(np.max(lens))
        f["sent_len_min"] = float(np.min(lens))
        f["short_sent_frac"] = float(np.mean(np.array(lens) < 4))
        f["long_sent_frac"] = float(np.mean(np.array(lens) >= 15))
    return f

In [9]:
# ---------------- 3. Grammar-error features ----------------
LT_CATS = ["GRAMMAR", "TYPOS", "CONFUSED_WORDS", "SEMANTICS", "REDUNDANCY", "COMPOUNDING",
           "COLLOCATIONS", "NONSTANDARD_PHRASES", "MISC"]
LT_IGNORE_FOR_CORE = {"PUNCTUATION", "CASING", "TYPOGRAPHY", "STYLE"}   # Whisper decides these, not the speaker


def languagetool_features(text, n_words):
    if not LT_OK or not text.strip():
        return {}
    try:
        matches = lt_tool.check(text)
    except Exception:
        return {}
    cats = [str(getattr(m, "category", "UNK")).upper() for m in matches]
    rules = [str(getattr(m, "ruleId", "")) for m in matches]
    nw = max(n_words, 1)
    core = [c for c in cats if c not in LT_IGNORE_FOR_CORE]
    f = {
        "lt_total": len(matches),
        "lt_total_per100": 100.0 * len(matches) / nw,
        "lt_core": len(core),
        "lt_core_per100": 100.0 * len(core) / nw,
        "lt_unique_rules": len(set(rules)),
    }
    for c in LT_CATS:
        k = sum(1 for x in cats if x == c)
        f[f"lt_{c.lower()}"] = k
        f[f"lt_{c.lower()}_per100"] = 100.0 * k / nw
    return f


@torch.no_grad()
def cola_scores(sentences):
    if not sentences:
        return []
    enc = cola_tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=128).to(DEVICE)
    probs = torch.softmax(cola_model(**enc).logits, dim=-1)[:, 1]   # label 1 = acceptable
    return probs.float().cpu().tolist()


@torch.no_grad()
def sentence_nlls(sentences):
    out = []
    bos = torch.full((1, 1), gpt_tok.bos_token_id, dtype=torch.long, device=DEVICE)
    for s in sentences:
        ids = gpt_tok(s, return_tensors="pt", truncation=True, max_length=200).input_ids.to(DEVICE)
        ids = torch.cat([bos, ids], dim=1)
        logits = gpt_model(ids).logits[0, :-1]
        loss = torch.nn.functional.cross_entropy(logits, ids[0, 1:], reduction="mean")
        out.append(loss.item())
    return out


def lm_grammar_features(text):
    f = {}
    sents = [s for s in sent_tokenize(text) if len(re.findall(r"[A-Za-z']+", s)) >= 3]
    if not sents:
        return f
    cs = np.array(cola_scores(sents))
    f["cola_mean"] = cs.mean()
    f["cola_min"] = cs.min()
    f["cola_std"] = cs.std()
    f["cola_frac_lt_05"] = float((cs < 0.5).mean())
    ns = np.array(sentence_nlls(sents))
    f["gpt_nll_mean"] = ns.mean()
    f["gpt_nll_max"] = ns.max()
    f["gpt_nll_median"] = float(np.median(ns))
    f["gpt_nll_std"] = ns.std()
    return f

In [10]:
# ---------------- 4. Small acoustic set + 5. Whisper encoder embedding ----------------
def acoustic_features(y, sr=16000):
    f = {"audio_duration": len(y) / sr}
    rms = librosa.feature.rms(y=y)[0]
    f["rms_mean"] = rms.mean()
    f["rms_std"] = rms.std()
    f["zcr_mean"] = librosa.feature.zero_crossing_rate(y)[0].mean()
    f["centroid_mean"] = librosa.feature.spectral_centroid(y=y, sr=sr)[0].mean()
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        f[f"mfcc_{i + 1}_mean"] = mfcc[i].mean()
    try:
        f0 = librosa.yin(y, fmin=70, fmax=400, sr=sr)
        f0 = f0[np.isfinite(f0)]
        f0 = f0[(f0 >= 70) & (f0 <= 400)]
        if len(f0):
            f["pitch_mean"] = f0.mean()
            f["pitch_std"] = f0.std()
            f["pitch_range"] = np.percentile(f0, 95) - np.percentile(f0, 5)
    except Exception:
        pass
    intervals = librosa.effects.split(y, top_db=30)
    f["voiced_fraction"] = sum(e - s for s, e in intervals) / max(len(y), 1)
    return f


@torch.no_grad()
def whisper_encoder_embedding(y):
    # Mean + std pooling over all valid encoder frames, processing the clip in 30 s chunks
    chunk = 30 * 16000
    n_mels = whisper_model.dims.n_mels
    dtype = next(whisper_model.encoder.parameters()).dtype
    sums, sqs, total = 0.0, 0.0, 0
    for s in range(0, len(y), chunk):
        seg = y[s:s + chunk]
        if s > 0 and len(seg) < 16000:       # ignore tiny tail (<1 s)
            continue
        n_valid = max(1, int(math.ceil(len(seg) / chunk * 1500)))
        mel = whisper.log_mel_spectrogram(whisper.pad_or_trim(seg), n_mels=n_mels).to(DEVICE).to(dtype)
        h = whisper_model.encoder(mel.unsqueeze(0))[0, :n_valid].float()
        sums = sums + h.sum(0)
        sqs = sqs + (h ** 2).sum(0)
        total += n_valid
    mean = sums / total
    std = (sqs / total - mean ** 2).clamp_min(0).sqrt()
    return torch.cat([mean, std]).cpu().numpy().astype(np.float32)

## 7. Per-file extraction with caching (failed files keep their row)

In [11]:
def extract_all(path):
    y, sr = librosa.load(str(path), sr=16000, mono=True)
    if len(y) == 0:
        raise ValueError("Empty audio")
    duration = len(y) / sr

    result = whisper_model.transcribe(
        y, language="en", fp16=(DEVICE == "cuda"), temperature=0.0,
        condition_on_previous_text=False, word_timestamps=True, verbose=None)
    text = result.get("text", "").strip()
    segments = result.get("segments", [])

    feats = {}
    feats.update(acoustic_features(y, sr))
    feats.update(whisper_features(segments, duration))
    feats.update(text_features(text, duration))
    n_words = feats.get("n_words", 0)
    feats.update(languagetool_features(text, n_words))
    feats.update(lm_grammar_features(text))

    emb = whisper_encoder_embedding(y)
    return feats, emb, text


def process_dataset(df, audio_dir, split_name):
    rows, failed = [], []
    for fname in tqdm(df["filename"].astype(str).tolist(), desc=split_name):
        cache_path = CACHE_DIR / f"{split_name}_{Path(fname).stem}.pkl"
        try:
            if cache_path.exists():
                with open(cache_path, "rb") as fh:
                    rec = pickle.load(fh)
            else:
                feats, emb, text = extract_all(Path(audio_dir) / fname)
                rec = {"feats": feats, "emb": emb, "text": text}
                with open(cache_path, "wb") as fh:
                    pickle.dump(rec, fh)
        except Exception as e:
            print(f"FAILED: {fname} -> {e}")
            failed.append((fname, str(e)))
            rec = {"feats": {}, "emb": None, "text": ""}
        rows.append(rec)
    return rows, failed

In [12]:
train_rows, train_failed = process_dataset(train_df, TRAIN_AUDIO_DIR, "train")
test_rows, test_failed = process_dataset(test_df, TEST_AUDIO_DIR, "test")
print("Failed train:", len(train_failed), "| Failed test:", len(test_failed))

train:   0%|          | 0/769 [00:00<?, ?it/s]

test:   0%|          | 0/216 [00:00<?, ?it/s]

Failed train: 0 | Failed test: 0


## 8. Assemble feature matrices

In [13]:
def stack_emb(rows):
    dim = next(r["emb"].shape[0] for r in rows if r["emb"] is not None)
    arr = np.full((len(rows), dim), np.nan, dtype=np.float32)
    for i, r in enumerate(rows):
        if r["emb"] is not None:
            arr[i] = r["emb"]
    return arr


y = train_df["label"].astype(float).values
assert not np.isnan(y).any(), "NaN in labels"

texts_tr = np.array([r["text"] for r in train_rows], dtype=object)
texts_te = np.array([r["text"] for r in test_rows], dtype=object)
print("Empty transcripts  train/test:", int((texts_tr == "").sum()), "/", int((texts_te == "").sum()))
print("Example transcript:", texts_tr[0][:300])

# ---------- tabular features ----------
tab_tr = pd.DataFrame([r["feats"] for r in train_rows])
tab_te = pd.DataFrame([r["feats"] for r in test_rows])

FEATURE_COLUMNS = [c for c in tab_tr.columns
                   if pd.api.types.is_numeric_dtype(tab_tr[c])
                   and tab_tr[c].notna().mean() >= 0.5
                   and tab_tr[c].nunique(dropna=True) > 1]
tab_te = tab_te.reindex(columns=FEATURE_COLUMNS)
tab_tr = tab_tr[FEATURE_COLUMNS]

tab_tr = tab_tr.replace([np.inf, -np.inf], np.nan)
tab_te = tab_te.replace([np.inf, -np.inf], np.nan)

train_medians = tab_tr.median()
tab_tr = tab_tr.fillna(train_medians)
tab_te = tab_te.fillna(train_medians)

# winsorise with TRAIN percentiles (protects Ridge/SVR from outliers)
lo, hi = tab_tr.quantile(0.005), tab_tr.quantile(0.995)
tab_tr = tab_tr.clip(lo, hi, axis=1)
tab_te = tab_te.clip(lo, hi, axis=1)

X_tab = tab_tr.values.astype(np.float64)
X_tab_te = tab_te.values.astype(np.float64)
print("Tabular features:", X_tab.shape, "| test:", X_tab_te.shape)
print("Grammar-error features kept:",
      [c for c in FEATURE_COLUMNS if c.startswith(("lt_", "cola_", "gpt_"))][:12], "...")

# ---------- Whisper-encoder embeddings ----------
E_wh = stack_emb(train_rows)
E_wh_te = stack_emb(test_rows)
col_mean = np.nanmean(E_wh, axis=0)
E_wh = np.where(np.isnan(E_wh), col_mean, E_wh).astype(np.float64)
E_wh_te = np.where(np.isnan(E_wh_te), col_mean, E_wh_te).astype(np.float64)
print("Whisper embedding:", E_wh.shape)

# ---------- sentence-transformer embeddings of the transcript ----------
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)
E_tx = st_model.encode(texts_tr.tolist(), batch_size=32, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
E_tx_te = st_model.encode(texts_te.tolist(), batch_size=32, normalize_embeddings=True, show_progress_bar=False).astype(np.float64)
del st_model
gc.collect()
print("Text embedding:", E_tx.shape)

Empty transcripts  train/test: 4 / 1
Example transcript: The plate round is very, very beautiful and fun. There are, there are plates like the marigold round, the swing. The students usually enjoy the swing. It has to be like, it's more like a game of fairs, because one sits in it and the other sits in it. We'll have to push and move around, which is fine
Tabular features: (769, 132) | test: (216, 132)
Grammar-error features kept: ['lt_total', 'lt_total_per100', 'lt_core', 'lt_core_per100', 'lt_unique_rules', 'lt_grammar', 'lt_grammar_per100', 'lt_typos', 'lt_typos_per100', 'lt_confused_words', 'lt_confused_words_per100', 'lt_redundancy'] ...
Whisper embedding: (769, 1536)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Text embedding: (769, 768)


## 9. Cross-validation setup

Repeated stratified 5-fold (x3). Every base model is evaluated and used the same way:
out-of-fold predictions are averaged across repeats, and **each fold model also predicts the test set** (test predictions averaged).
This removes the old "CV config vs. final config" mismatch.

In [14]:
y_bins = pd.cut(y, bins=[-np.inf, 1.5, 2.5, 3.5, 4.5, np.inf], labels=False).astype(int)
bin_counts = np.bincount(y_bins)
print("Target band counts:", bin_counts.tolist())

if bin_counts[bin_counts > 0].min() >= N_SPLITS:
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    splits = list(cv.split(X_tab, y_bins))
    print("Using RepeatedStratifiedKFold")
else:
    cv = RepeatedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    splits = list(cv.split(X_tab))
    print("Rare target band -> using RepeatedKFold")
print("Total fold fits per model:", len(splits))


def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))


def evaluate(y_true, pred, name):
    r = {"model": name, "RMSE": rmse(y_true, pred), "MAE": float(mean_absolute_error(y_true, pred)),
         "Pearson": float(pearsonr(y_true, pred)[0]), "Spearman": float(spearmanr(y_true, pred)[0])}
    print(f"{name:<22} RMSE {r['RMSE']:.4f} | MAE {r['MAE']:.4f} | Pearson {r['Pearson']:.4f} | Spearman {r['Spearman']:.4f}")
    return r


def run_cv(make_model, Xtr, Xte, y, splits):
    n = len(y)
    oof, cnt, te = np.zeros(n), np.zeros(n), np.zeros(len(Xte))
    for tr, va in splits:
        m = make_model()
        m.fit(Xtr[tr], y[tr])
        oof[va] += m.predict(Xtr[va])
        cnt[va] += 1
        te += m.predict(Xte)
    return oof / cnt, te / len(splits)


Y_MIN, Y_MAX = float(y.min()), float(y.max())
print("Label range:", Y_MIN, "-", Y_MAX, "| std:", round(float(y.std()), 4),
      "| RMSE of predicting the mean:", round(float(y.std()), 4))

Target band counts: [41, 181, 238, 176, 133]
Using RepeatedStratifiedKFold
Total fold fits per model: 15
Label range: 0.0 - 5.0 | std: 1.2382 | RMSE of predicting the mean: 1.2382


## 10. Base models

In [15]:
def make_xgb():
    return XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.03, subsample=0.8,
                        colsample_bytree=0.6, min_child_weight=3, reg_lambda=5.0,
                        objective="reg:squarederror", random_state=SEED, n_jobs=-1)


def make_et():
    return ExtraTreesRegressor(n_estimators=500, max_features=0.5, min_samples_leaf=2,
                               random_state=SEED, n_jobs=-1)


def make_ridge(lo=0, hi=5):
    return Pipeline([("sc", StandardScaler()), ("m", RidgeCV(alphas=np.logspace(lo, hi, 15)))])


def make_svr():
    return GridSearchCV(
        Pipeline([("sc", StandardScaler()), ("m", SVR(kernel="rbf", epsilon=0.1))]),
        {"m__C": [0.3, 1, 3, 10]}, cv=3, scoring="neg_root_mean_squared_error")


def make_tfidf():
    return GridSearchCV(
        Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=15000)),
                  ("m", Ridge())]),
        {"m__alpha": [1, 3, 10, 30]}, cv=3, scoring="neg_root_mean_squared_error")


experiments = [
    ("xgb_tab",       make_xgb,                    X_tab, X_tab_te),
    ("et_tab",        make_et,                     X_tab, X_tab_te),
    ("ridge_tab",     lambda: make_ridge(-1, 4),   X_tab, X_tab_te),
    ("ridge_whisper", lambda: make_ridge(1, 5),    E_wh,  E_wh_te),
    ("svr_whisper",   make_svr,                    E_wh,  E_wh_te),
    ("ridge_textemb", lambda: make_ridge(0, 4),    E_tx,  E_tx_te),
    ("svr_textemb",   make_svr,                    E_tx,  E_tx_te),
    ("tfidf_ridge",   make_tfidf,                  texts_tr, texts_te),
]

oof_preds, test_preds, score_rows = {}, {}, []
for name, factory, Xtr_, Xte_ in tqdm(experiments, desc="base models"):
    oof_preds[name], test_preds[name] = run_cv(factory, Xtr_, Xte_, y, splits)
    score_rows.append(evaluate(y, np.clip(oof_preds[name], Y_MIN, Y_MAX), name))

base models:   0%|          | 0/8 [00:00<?, ?it/s]

xgb_tab                RMSE 0.6106 | MAE 0.4768 | Pearson 0.8706 | Spearman 0.8084
et_tab                 RMSE 0.6436 | MAE 0.5089 | Pearson 0.8571 | Spearman 0.7900
ridge_tab              RMSE 0.6635 | MAE 0.5237 | Pearson 0.8444 | Spearman 0.7901
ridge_whisper          RMSE 0.5656 | MAE 0.4275 | Pearson 0.8913 | Spearman 0.8484
svr_whisper            RMSE 0.5726 | MAE 0.4322 | Pearson 0.8922 | Spearman 0.8561
ridge_textemb          RMSE 0.9426 | MAE 0.7180 | Pearson 0.6491 | Spearman 0.6644
svr_textemb            RMSE 0.9305 | MAE 0.6954 | Pearson 0.6613 | Spearman 0.6751
tfidf_ridge            RMSE 1.0500 | MAE 0.7966 | Pearson 0.5644 | Spearman 0.6239


### Optional: fine-tune DeBERTa-v3 on the transcripts
Set `RUN_DEBERTA = True` in the config cell to enable. Uses one repeat (5 folds) of the same splits.

In [16]:
def run_deberta(texts_train, y, texts_test, splits_one, model_name="microsoft/deberta-v3-base",
                epochs=4, lr=2e-5, bs=16, max_len=256):
    from transformers import get_linear_schedule_with_warmup
    tok = AutoTokenizer.from_pretrained(model_name)
    mu, sd = float(y.mean()), float(y.std())
    yn = (y - mu) / sd

    def enc(txts):
        return tok(list(txts), truncation=True, max_length=max_len, padding=True, return_tensors="pt")

    @torch.no_grad()
    def predict(model, txts):
        model.eval()
        out = []
        for i in range(0, len(txts), 32):
            e = enc(txts[i:i + 32]).to(DEVICE)
            out.append(model(**e).logits.squeeze(-1).float().cpu().numpy())
        return np.concatenate(out)

    oof, te = np.zeros(len(y)), np.zeros(len(texts_test))
    for k, (tr, va) in enumerate(splits_one):
        torch.manual_seed(SEED + k)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        steps = epochs * math.ceil(len(tr) / bs)
        sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        for ep in range(epochs):
            model.train()
            perm = np.random.RandomState(SEED + 100 * k + ep).permutation(tr)
            for i in range(0, len(perm), bs):
                idx = perm[i:i + bs]
                e = enc(texts_train[idx]).to(DEVICE)
                pred = model(**e).logits.squeeze(-1)
                target = torch.tensor(yn[idx], dtype=torch.float32, device=DEVICE)
                loss = torch.nn.functional.mse_loss(pred, target)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad()
        oof[va] = predict(model, texts_train[va])
        te += predict(model, texts_test)
        print(f"DeBERTa fold {k + 1}/{len(splits_one)} RMSE: {rmse(y[va], oof[va] * sd + mu):.4f}")
        del model, opt
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return oof * sd + mu, te / len(splits_one) * sd + mu


if RUN_DEBERTA:
    oof_preds["deberta"], test_preds["deberta"] = run_deberta(texts_tr, y, texts_te, splits[:N_SPLITS])
    score_rows.append(evaluate(y, np.clip(oof_preds["deberta"], Y_MIN, Y_MAX), "deberta"))
else:
    print("DeBERTa skipped (RUN_DEBERTA = False)")

DeBERTa skipped (RUN_DEBERTA = False)


## 11. Stacking
Non-negative linear regression (with intercept) on the out-of-fold predictions replaces the hand-picked 70/20/10 weights.
The stacking CV score below is slightly optimistic (the base OOF predictions were produced on the same data), but it is a fair way to compare against a simple average.

In [21]:
names = list(oof_preds.keys())
oof_mat = np.column_stack([oof_preds[n] for n in names])
te_mat = np.column_stack([test_preds[n] for n in names])

# simple average baseline
avg_oof = oof_mat.mean(axis=1)
evaluate(y, np.clip(avg_oof, Y_MIN, Y_MAX), "simple_average")

# stacked model: honest-ish CV estimate
stack_cv = np.zeros(len(y))
for tr, va in KFold(5, shuffle=True, random_state=SEED).split(oof_mat):
    s = LinearRegression(positive=True).fit(oof_mat[tr], y[tr])
    stack_cv[va] = s.predict(oof_mat[va])
stack_score = evaluate(y, np.clip(stack_cv, Y_MIN, Y_MAX), "stacked (CV)")

# final stacker on all OOF predictions
stacker = LinearRegression(positive=True).fit(oof_mat, y)
print("\nStacker weights:")
for n, w in sorted(zip(names, stacker.coef_), key=lambda t: -t[1]):
    print(f"  {n:<16} {w:.3f}")
print("  intercept        ", round(float(stacker.intercept_), 3))

summary = pd.DataFrame(score_rows + [stack_score]).sort_values("RMSE").reset_index(drop=True)
display(summary)

test_pred_final = np.clip(stacker.predict(te_mat), Y_MIN, Y_MAX)
print("\nTrain label  mean/std:", round(float(y.mean()), 3), round(float(y.std()), 3))
print("Test pred    mean/std:", round(float(test_pred_final.mean()), 3), round(float(test_pred_final.std()), 3))
if abs(test_pred_final.mean() - y.mean()) > 0.3:
    print("WARNING: test prediction mean differs a lot from train label mean - check for train/test mismatch.")

simple_average         RMSE 0.6347 | MAE 0.5120 | Pearson 0.8927 | Spearman 0.8597
stacked (CV)           RMSE 0.5275 | MAE 0.3946 | Pearson 0.9052 | Spearman 0.8655

Stacker weights:
  svr_whisper      0.412
  xgb_tab          0.284
  ridge_whisper    0.273
  ridge_textemb    0.084
  et_tab           0.054
  ridge_tab        0.043
  svr_textemb      0.018
  tfidf_ridge      0.000
  intercept         -0.56


,model,RMSE,MAE,Pearson,Spearman
0,stacked (CV),0.527518,0.394556,0.905187,0.865494
1,ridge_whisper,0.565648,0.427520,0.891273,0.848419
2,svr_whisper,0.572569,0.432191,0.892167,0.856103
3,xgb_tab,0.610558,0.476805,0.870583,0.808400
4,et_tab,0.643636,0.508933,0.857085,0.789998
5,ridge_tab,0.663469,0.523674,0.844444,0.790119
6,svr_textemb,0.930523,0.695364,0.661349,0.675149
7,ridge_textemb,0.942621,0.718001,0.649130,0.664362
8,tfidf_ridge,1.049962,0.796645,0.564375,0.623901



Train label  mean/std: 3.313 1.238
Test pred    mean/std: 3.278 0.772


## 12. Final submission (single file)

In [25]:
id_col, label_col = sample_sub.columns[0], sample_sub.columns[1]

# Create a mapping of filename to its predicted score
pred_map = dict(zip(test_df["filename"].astype(str).values, test_pred_final))

# Map predictions to the sample submission file names to ensure correct order and length
submission = pd.DataFrame({
    id_col: sample_sub[id_col].astype(str).values,
    label_col: [pred_map.get(f, Y_MIN) for f in sample_sub[id_col].astype(str).values]
})

assert len(submission) == len(sample_sub), "Row count mismatch"
assert set(submission[id_col]) == set(sample_sub[id_col].astype(str)), "Filenames differ from sample_submission"
assert submission[label_col].notna().all(), "NaN predictions"
assert submission[label_col].between(Y_MIN, Y_MAX).all(), "Out-of-range predictions"

SUBMISSION_PATH = "/content/submission_v3.csv"
submission.to_csv(SUBMISSION_PATH, index=False)
submission.to_csv(DATA_ROOT / "submission_v3.csv", index=False)

print("Saved:", SUBMISSION_PATH, "| copy:", DATA_ROOT / "submission_v3.csv")
print("Failed test files (imputed):", [f for f, _ in test_failed])
display(submission.head(10))
print(submission[label_col].describe())

Saved: /content/submission_v3.csv | copy: /content/drive/MyDrive/data/submission_v3.csv
Failed test files (imputed): []


,filename,label
0,audio_804.wav,0.000000
1,audio_1028.wav,0.000000
2,audio_865.wav,0.000000
3,audio_774.wav,0.000000
4,audio_1138.wav,0.000000
5,audio_278.wav,0.000000
6,audio_1212.wav,0.000000
7,audio_178.wav,3.620899
8,audio_542.wav,0.000000
9,audio_248.wav,0.000000


count    204.000000
mean       0.404139
std        1.108849
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        4.891656
Name: label, dtype: float64


In [29]:
# Save predictions for all 216 files present in test_df
all_test_submission = pd.DataFrame({
    "filename": test_df["filename"].astype(str).values,
    "label": test_pred_final
})

ALL_SUBMISSION_PATH = "/content/submission_all_216.csv"
all_test_submission.to_csv(ALL_SUBMISSION_PATH, index=False)
all_test_submission.to_csv(DATA_ROOT / "submission_all_216.csv", index=False)

print(f"Saved full predictions for all {len(all_test_submission)} files to:")
print(f"  - {ALL_SUBMISSION_PATH}")
print(f"  - {DATA_ROOT / 'submission_all_216.csv'}")

display(all_test_submission.tail(10))
print(all_test_submission["label"].describe())

Saved full predictions for all 216 files to:
  - /content/submission_all_216.csv
  - /content/drive/MyDrive/data/submission_all_216.csv


,filename,label
206,audio_20.wav,2.791034
207,audio_124.wav,2.972807
208,audio_54.wav,2.831445
209,audio_45.wav,5.000000
210,audio_71.wav,4.766875
211,audio_184.wav,2.827240
212,audio_24.wav,2.036115
213,audio_140.wav,3.730865
214,audio_165.wav,3.000018
215,audio_170.wav,2.524464


count    216.000000
mean       3.278228
std        0.773363
min        1.746999
25%        2.671737
50%        3.199047
75%        3.735821
max        5.000000
Name: label, dtype: float64


In [28]:
# Save the reusable pieces (fold models are not stored; test predictions already come from them)
with open(DATA_ROOT / "pipeline_v3.pkl", "wb") as fh:
    pickle.dump({
        "whisper_size": WHISPER_SIZE,
        "feature_columns": FEATURE_COLUMNS,
        "train_medians": train_medians,
        "clip_lo": lo, "clip_hi": hi,
        "base_model_names": names,
        "stacker": stacker,
        "label_range": (Y_MIN, Y_MAX),
        "oof_preds": oof_preds,
        "test_preds": test_preds,
        "cv_summary": summary,
    }, fh)
print("Pipeline metadata saved.")

try:
    from google.colab import files
    files.download(SUBMISSION_PATH)
except Exception as e:
    print("Download skipped:", e)

Pipeline metadata saved.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>